# The ₹99 Crore Question
## One raw file → a live dashboard

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

Nine stages. Two million UPI transactions. Everything you are about to see is on the
syllabus you already have.

| # | Stage | Taught on |
|---|---|---|
| 0 | The raw file lands | Day 4 · Day 12 |
| 1 | Bronze — raw made durable | Day 6 ✅ · Day 10 |
| 2 | Silver — raw made trustworthy | **Days 6–7 ✅** |
| 3 | Gold — trustworthy made useful | **Day 7 ✅** |
| 4 | Time travel | Day 10 · Day 24 |
| 5 | Streaming ingestion | Day 12 · Day 13 |
| 6 | Scheduled jobs | Day 14 |
| 7 | Dashboard | Day 15 |
| 8 | Snowflake | Days 16–25 |

✅ = already covered.

## Configuration

Unity Catalog schema, volume and table names for this session.

In [ ]:
CATALOG = "workspace"
SCHEMA  = "showcase_d8"
VOLUME  = "showcase_files"

SOURCE_TABLE = "workspace.default.upi_transactions_2026"

VOL_ROOT   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SOURCE_CSV = f"{VOL_ROOT}/upi_transactions_2026.csv"
LANDING    = f"{VOL_ROOT}/landing"
CHECKPOINT = f"{VOL_ROOT}/_checkpoints"
SCHEMA_LOC = f"{VOL_ROOT}/_schema"

T_BRONZE   = f"{CATALOG}.{SCHEMA}.upi_bronze"
T_SILVER   = f"{CATALOG}.{SCHEMA}.upi_silver"
T_GOLD_BNK = f"{CATALOG}.{SCHEMA}.upi_gold_bank_health"
T_GOLD_HR  = f"{CATALOG}.{SCHEMA}.upi_gold_hourly_pulse"
T_STREAM   = f"{CATALOG}.{SCHEMA}.upi_stream_bronze"

import time
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, DecimalType)

# The schema is declared, not inferred. Declaring it removes the inference pass —
# which means the CSV is being given every advantage in the comparison ahead.
CSV_SCHEMA = StructType([
    StructField("txn_id",     StringType()),
    StructField("txn_time",   StringType()),
    StructField("amount_inr", DoubleType()),
    StructField("bank",       StringType()),
    StructField("city",       StringType()),
    StructField("status",     StringType()),
])

SRC_COLS = ["txn_id", "txn_time", "amount_inr", "bank", "city", "status"]

print(f"Source table : {SOURCE_TABLE}")
print(f"Volume root  : {VOL_ROOT}")

## Workspace objects

A Unity Catalog schema and a Volume. On Free Edition, Volumes are the storage path —
DBFS root and mounts are deprecated for new accounts, though the filesystem still
underpins the platform.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

for _p in (LANDING, CHECKPOINT, SCHEMA_LOC):
    dbutils.fs.mkdirs(_p)

for _p in (VOL_ROOT, LANDING, CHECKPOINT, SCHEMA_LOC):
    print(f"ready: {_p}")

## The source file

Our transactions currently live in a Unity Catalog table, which is already columnar and
already has a transaction log. To start where a real pipeline starts, we write them out
as a plain CSV first.

In [ ]:
_tmp = f"{VOL_ROOT}/_export_tmp"

(spark.table(SOURCE_TABLE)
 .select(*SRC_COLS)
 .coalesce(1)
 .write.mode("overwrite")
 .option("header", True)
 .csv(_tmp))

_part = [f.path for f in dbutils.fs.ls(_tmp) if f.name.endswith(".csv")][0]
dbutils.fs.cp(_part, SOURCE_CSV)
dbutils.fs.rm(_tmp, recurse=True)

display(dbutils.fs.ls(VOL_ROOT))

In [ ]:
_chk = spark.read.option("header", True).schema(CSV_SCHEMA).csv(SOURCE_CSV)

print(f"Rows: {_chk.count():,}")
display(_chk.agg(
    F.sum("amount_inr").cast(DecimalType(20, 2)).alias("total_value_inr"),
    F.sum((F.col("status") == "FAILED").cast("int")).alias("failed_txns"),
))

In [ ]:
display(_chk.groupBy("city").count().orderBy(F.desc("count")).limit(5))

In [ ]:
display(_chk.groupBy("bank").count().orderBy(F.desc("count")).limit(5))

---
# STAGE 0 · The raw file lands
### *"Somebody's payment system wrote this file last night. Nobody has looked at it yet."*

**Portability:** UNIVERSAL — every platform starts here · the storage path is DATABRICKS
**Taught on:** Day 4 · Day 12

The file is **inert**. It is flat text in object storage. It has no schema anyone agreed
to, no write guarantees, no history, and nobody can ask it a question without re-parsing
all two million rows.

Every stage that follows exists to fix exactly one of those four things.

In [ ]:
display(dbutils.fs.ls(VOL_ROOT))

### What one question costs

A real business question — total value by bank — not a row count. That distinction
matters: `.count()` on a Delta table is answered from the transaction log's statistics
without touching the data, so counting would flatter Delta unfairly. An aggregate forces
both sides to actually read.

In [ ]:
_t0 = time.time()
_csv_res = (spark.read
            .option("header", True)
            .schema(CSV_SCHEMA)
            .csv(SOURCE_CSV)
            .groupBy("bank")
            .agg(F.count("*").alias("n"),
                 F.sum("amount_inr").alias("total"))
            .orderBy(F.desc("total"))
            .collect())
_csv_secs = round(time.time() - _t0, 1)

print(f"Rows scanned : {sum(r['n'] for r in _csv_res):,}")
print(f"Seconds      : {_csv_secs}")

---
# STAGE 1 · Bronze — raw made durable
### *"Same data. One difference: it now remembers."*

**Portability:** Delta Lake is open source / SPARK · UC managed tables are DATABRICKS
**Taught on:** Day 6 (the CSV read) ✅ · Day 10 (Delta)

Bronze is deliberately dumb: land the raw data as-is, in a columnar, transaction-logged
format, and change nothing else. No cleaning, no renaming, no business logic.

When cleaning logic later turns out to be wrong — and it will — bronze is what you replay
from. That is precisely why you don't clean it. Bronze is not a backup: backups are for
disasters, bronze is for the ordinary weekly event of discovering your business logic was
wrong.

**Industry example:** every payments company runs a bronze layer for one reason — a
regulator can ask "what did the switch actually send you on 14 August?", and "we overwrote
it during cleaning" is not an acceptable answer.

In [ ]:
bronze_df = (spark.read
             .option("header", True)
             .schema(CSV_SCHEMA)
             .csv(SOURCE_CSV)
             .withColumn("_ingest_ts",   F.current_timestamp())
             .withColumn("_source_file", F.col("_metadata.file_path")))

(bronze_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(T_BRONZE))

print(f"Bronze table written: {T_BRONZE}")

### The same question, against Delta

Identical question. Identical row count. Nothing dropped.

Watch the **size**, not the clock. A text file stores every value as characters, row by
row, so a question about two columns has to read all six. Delta stores each column
separately and compresses it.

In [ ]:
_t0 = time.time()
_delta_res = (spark.table(T_BRONZE)
              .groupBy("bank")
              .agg(F.count("*").alias("n"),
                   F.sum("amount_inr").alias("total"))
              .orderBy(F.desc("total"))
              .collect())
_delta_secs = round(time.time() - _t0, 1)

_csv_bytes   = [f.size for f in dbutils.fs.ls(VOL_ROOT) if f.name.endswith(".csv")][0]
_delta_bytes = spark.sql(f"DESCRIBE DETAIL {T_BRONZE}").collect()[0]["sizeInBytes"]

print(f"Rows scanned : {sum(r['n'] for r in _delta_res):,}   (both sides)")
print()
print("TIME")
print(f"  CSV   : {_csv_secs}s")
print(f"  Delta : {_delta_secs}s")
print()
print("SIZE ON DISK — same two million transactions, same six columns")
print(f"  CSV   : {_csv_bytes/1024/1024:,.0f} MB")
print(f"  Delta : {_delta_bytes/1024/1024:,.0f} MB")
print(f"  Delta holds the same data in {round(100.0*_delta_bytes/_csv_bytes)}% of the space")
print(f"  Compression : {round(_csv_bytes/_delta_bytes, 1)}x")

---
# STAGE 2 · Silver — raw made trustworthy
### *"This is the stage where you already know the code."*

**Portability:** UNIVERSAL concept · the API here is SPARK
**Taught on:** **Days 6 and 7** ✅

Silver does the actual work of turning raw data into something you would bet money on,
and it is built entirely from `select`, `filter`, `withColumn`, `cast` and
`dropDuplicates`.

This is also where every real production bug lives. Anyone can call `RESTORE`. Almost
nobody gets the cleaning right.

**On timezones:** `txn_time` carries a `Z` suffix, so it parses as UTC. We do not convert
to IST here — the source data starts its day at `00:00`, so shifting it would push the day
to a 5:30 a.m. start. The timestamps are already local; the `Z` is an artefact of how the
data was generated. That is itself a data quality lesson.

In [ ]:
silver_typed = (spark.table(T_BRONZE)
                .select(
                    F.col("txn_id").alias("txn_id"),
                    F.col("txn_time").cast("timestamp").alias("txn_ts"),
                    F.col("amount_inr").cast(DecimalType(18, 2)).alias("amount_inr"),
                    F.upper(F.trim(F.col("status"))).alias("status"),
                    F.initcap(F.trim(F.col("city"))).alias("city"),
                    F.upper(F.trim(F.col("bank"))).alias("bank"),
                ))

# Verify the parse BEFORE filtering. The filter below removes rows with a null
# timestamp — so if the format failed to parse, the filter would silently delete
# the dataset. Measure first, filter second.
_parse = silver_typed.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("txn_ts").isNull().cast("int")).alias("bad_timestamps"),
    F.sum(F.col("txn_id").isNull().cast("int")).alias("missing_ids"),
).collect()[0]

print(f"Rows           : {_parse['rows']:,}")
print(f"Bad timestamps : {_parse['bad_timestamps']:,}")
print(f"Missing txn_id : {_parse['missing_ids']:,}")

In [ ]:
silver_df = (silver_typed
             # A row with no id or no timestamp cannot be reasoned about
             .filter(F.col("txn_id").isNotNull() & F.col("txn_ts").isNotNull())
             # The same transaction can arrive twice from a retrying switch
             .dropDuplicates(["txn_id"])
             # Derived columns the gold layer will group by
             .withColumn("txn_hour",  F.hour("txn_ts"))
             .withColumn("txn_date",  F.to_date("txn_ts"))
             .withColumn("is_failed", (F.col("status") == F.lit("FAILED")).cast("int")))

(silver_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(T_SILVER))

print(f"Silver table written: {T_SILVER}")
display(spark.sql(f"SELECT COUNT(*) AS silver_rows FROM {T_SILVER}"))

### Why `DECIMAL`, and not `FLOAT`

One line in the cell above matters more than the rest, and it is the boring one:
`DecimalType`.

The common claim is that storing money as a float loses money in the total. Let's test
that properly, by measuring three different things:

| Measure | The question it answers |
|---|---|
| Aggregate drift | Would the management report look wrong? |
| Stored value wrong | Is the number in storage the number we were given? |
| Wrong after rounding to paise | Would anyone looking at a screen see a wrong number? |

In [ ]:
probe = (spark.table(T_BRONZE)
         .select(
             F.col("amount_inr").cast(DecimalType(18, 2)).alias("as_decimal"),
             F.col("amount_inr").cast("float").alias("as_float"),
         )
         .select(
             "as_decimal",
             F.col("as_decimal").cast(DecimalType(24, 8)).alias("decimal_exact"),
             F.col("as_float").cast(DecimalType(24, 8)).alias("float_exact"),
             F.col("as_float").cast(DecimalType(18, 2)).alias("float_rounded"),
             F.col("as_float").alias("as_float"),
         ))

_agg = probe.agg(
    F.sum("as_decimal").alias("sum_decimal"),
    F.sum("as_float").cast(DecimalType(24, 6)).alias("sum_float"),
).collect()[0]

_rows = probe.agg(
    F.count("*").alias("rows"),
    F.sum((F.col("float_exact") != F.col("decimal_exact")).cast("int")).alias("stored_wrong"),
    F.sum((F.col("float_rounded") != F.col("as_decimal")).cast("int")).alias("displays_wrong"),
).collect()[0]

n = _rows["rows"]
print("1. AGGREGATE — what an auditor checks")
print(f"     SUM as DECIMAL     : {_agg['sum_decimal']}")
print(f"     SUM as FLOAT       : {_agg['sum_float']}")
print(f"     Drift              : {_agg['sum_decimal'] - _agg['sum_float']}")
print()
print("2. STORED VALUE — is it the number we were given?")
print(f"     Rows               : {n:,}")
print(f"     Stored incorrectly : {_rows['stored_wrong']:,}  ({round(100.0*_rows['stored_wrong']/n,1)}%)")
print()
print("3. AFTER ROUNDING TO PAISE — what appears on a screen")
print(f"     Still incorrect    : {_rows['displays_wrong']:,}  ({round(100.0*_rows['displays_wrong']/n,1)}%)")

### The same rupee amount, twice

Two columns, side by side, holding what is supposed to be one number.

In [ ]:
display(probe
        .filter(F.col("float_exact") != F.col("decimal_exact"))
        .select("decimal_exact", "float_exact", "as_decimal", "float_rounded")
        .limit(15))

### Now ask the database for one of them

Every screen shows the right amount. So ask for a specific amount and see what comes back.

In [ ]:
_target = "432.90"

_n_dec = probe.filter(F.col("as_decimal") == F.lit(_target).cast(DecimalType(18, 2))).count()
_n_flt = probe.filter(F.col("float_exact") == F.lit(_target).cast(DecimalType(24, 8))).count()

print(f"Transactions worth exactly Rs {_target}")
print(f"  Stored as DECIMAL : {_n_dec:,} rows found")
print(f"  Stored as FLOAT   : {_n_flt:,} rows found")

---
# STAGE 3 · Gold — trustworthy made useful
### *"Two million rows become the handful of numbers a manager acts on."*

**Portability:** UNIVERSAL
**Taught on:** **Day 7** — `groupBy` and `agg` ✅

Gold is the layer with opinions. Bronze and silver are about fidelity; gold is about a
decision somebody has to make before lunch. Which bank carries our volume? What does the
day look like? Gold tables are small, cheap, and shaped like the question rather than
shaped like the source system.

**Industry example:** an NPCI-facing ops team does not open a notebook at 9 a.m. They open
one screen with a handful of tiles on it. Those tiles are gold tables.

In [ ]:
gold_bank = (spark.table(T_SILVER)
             .groupBy("bank")
             .agg(
                 F.count("*").alias("txn_count"),
                 F.sum("amount_inr").alias("total_inr"),
                 F.sum("is_failed").alias("failed_count"),
                 F.round(100.0 * F.sum("is_failed") / F.count("*"), 2).alias("failure_rate_pct"),
             )
             .orderBy(F.desc("total_inr")))

(gold_bank.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(T_GOLD_BNK))

display(spark.table(T_GOLD_BNK))

In [ ]:
# Grouped by hour of day only, across every date in the dataset.
# Grouping by (txn_date, txn_hour) gives 2,160 rows, which is not a readable chart.
gold_hour = (spark.table(T_SILVER)
             .groupBy("txn_hour")
             .agg(
                 F.count("*").alias("txn_count"),
                 F.sum("amount_inr").alias("total_inr"),
                 F.round(100.0 * F.sum("is_failed") / F.count("*"), 2).alias("failure_rate_pct"),
             )
             .orderBy("txn_hour"))

(gold_hour.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(T_GOLD_HR))

print(f"Rows: {spark.table(T_GOLD_HR).count()}")
display(spark.table(T_GOLD_HR))

### Two million transactions, in one row

In [ ]:
display(spark.sql(f"""
  SELECT
    COUNT(*)                                   AS total_txns,
    SUM(amount_inr)                            AS total_value_inr,
    SUM(is_failed)                             AS failed_txns,
    ROUND(100.0 * SUM(is_failed)/COUNT(*), 2)  AS failure_rate_pct
  FROM {T_SILVER}
"""))

---
# STAGE 4 · Time travel
### *"Every one of you will do this to a production table one day."*

**Portability:** DELTA LAKE (open source) — the concept exists in Snowflake too
**Taught on:** **Day 10** · Snowflake's version on **Day 24**

A Delta table keeps a record of every write made to it — who, when, and what — without
anyone asking it to. That record is what makes the next few cells possible.

In [ ]:
PRE_DELETE_VERSION = (spark.sql(f"DESCRIBE HISTORY {T_SILVER}")
                      .agg(F.max("version")).collect()[0][0])

print(f"Silver is currently at version {PRE_DELETE_VERSION}")
display(spark.sql(f"SELECT COUNT(*) AS rows_now, SUM(amount_inr) AS value_now FROM {T_SILVER}"))

### A junior engineer meant to delete test transactions from one city

They forgot the second half of the WHERE clause.

In [ ]:
spark.sql(f"DELETE FROM {T_SILVER} WHERE city = 'Hyderabad'")

display(spark.sql(f"SELECT COUNT(*) AS rows_now, SUM(amount_inr) AS value_now FROM {T_SILVER}"))

### The record nobody knew was being kept

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T_SILVER}"))

### The table as it was, before the mistake

Not a copy. The same table, at a previous point in time.

In [ ]:
display(spark.sql(f"""
  SELECT COUNT(*) AS rows_then, SUM(amount_inr) AS value_then
  FROM {T_SILVER} VERSION AS OF {PRE_DELETE_VERSION}
"""))

### One line

Nothing was backed up. Nothing was re-ingested.

Retention is finite and configurable, and `VACUUM` permanently removes old files — that
trade-off is Day 11.

In [ ]:
spark.sql(f"RESTORE TABLE {T_SILVER} TO VERSION AS OF {PRE_DELETE_VERSION}")

display(spark.sql(f"SELECT COUNT(*) AS rows_now, SUM(amount_inr) AS value_now FROM {T_SILVER}"))

---
# STAGE 5 · Streaming — the pipeline that never sleeps
### *"You never write the 'find the new rows' logic. That's the whole point."*

**Portability:** Structured Streaming is SPARK · Auto Loader (`cloudFiles`) is DATABRICKS
**Taught on:** **Day 12** (ingestion) · **Day 13** (streaming)
**FREE EDITION:** `Trigger.AvailableNow()` only — time-based triggers fail. The checkpoint
must live on a UC Volume.

Everything so far has been a full reload. That is fine at two million rows and completely
untenable at two billion.

Auto Loader keeps its own record of which files it has already consumed, so pointing it at
a directory and running it repeatedly processes only what is new — including files that
did not exist when the code was written. That bookkeeping is what a checkpoint is, and it
is the difference between a script and a pipeline.

In [ ]:
(spark.table(T_BRONZE).select(*SRC_COLS).limit(50_000)
 .coalesce(1).write.mode("overwrite")
 .option("header", True).csv(f"{LANDING}/batch_01"))

display(dbutils.fs.ls(LANDING))

In [ ]:
def run_autoloader():
    """Consume any new CSVs in LANDING into a streaming bronze table.
    availableNow: process everything available, then stop."""
    stream = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "csv")
              .option("cloudFiles.schemaLocation", SCHEMA_LOC)
              .option("header", True)
              .load(LANDING)
              .withColumn("_ingest_ts",   F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path")))

    q = (stream.writeStream
         .format("delta")
         .option("checkpointLocation", CHECKPOINT)
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable(T_STREAM))
    q.awaitTermination()
    return q

run_autoloader()
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T_STREAM}"))

### A new file arrives in the landing zone

In [ ]:
(spark.table(T_BRONZE).select(*SRC_COLS).limit(75_000)
 .withColumn("txn_id", F.concat(F.col("txn_id"), F.lit("-B02")))
 .coalesce(1).write.mode("overwrite")
 .option("header", True).csv(f"{LANDING}/batch_02"))

display(dbutils.fs.ls(LANDING))

### The same code again

Not one character changed.

In [ ]:
_before = spark.table(T_STREAM).count()
run_autoloader()
_after = spark.table(T_STREAM).count()

print(f"Rows before this run : {_before:,}")
print(f"Rows after this run  : {_after:,}")
print(f"Newly ingested       : {_after - _before:,}")

---
# STAGE 6 · Jobs — the pipeline that runs without you
### *"The difference between a notebook and a data platform is whether it runs at 6 a.m. when you're asleep."*

**Portability:** UNIVERSAL concept (Airflow, Snowflake Tasks) · the Workflows UI is DATABRICKS
**Taught on:** **Day 14** · orchestration Day 27 · monitoring and alerting Day 28
**FREE EDITION:** serverless jobs, maximum 5 concurrent job tasks per account.

A pipeline nobody has to be awake for is the minimum bar for production. The harder
question — what happens when it fails at 6 a.m. and nobody notices until 11 — is Day 28.

---
# STAGE 7 · Dashboard — what the business actually sees
### *"Nobody in the bank will ever open this notebook. They will open this."*

**Portability:** DATABRICKS AI/BI · the concept is UNIVERSAL
**Taught on:** the surface around **Day 15**; the gold tables behind it are Days 10–14

Every tile on a dashboard is a query against a gold table. That is why the gold layer is
shaped like the question and not like the source system — and it is the only part of the
pipeline the business will ever judge you on.

---
# STAGE 8 · Snowflake — where the analysts live
### *"You are not building this today. You are building it on Day 16."*

**Taught on:** **Days 16–24** · integration on **Day 25**

Databricks is where data is engineered. Snowflake is where a very large number of analysts
actually query it. Knowing both, and knowing how to move data between them, is the point
of this course.

### Do not create a Snowflake account yet

We create accounts together, in class, on **Day 16**. The trial runs for 30 days and it has
to still be alive when you need it for the capstone. Signing up today means it expires
before your project does.

---
# The map
### *"Nothing you just saw is magic. All of it is on the syllabus."*

| # | Stage | Taught on |
|---|---|---|
| 0 | The raw file lands | Day 4 · Day 12 |
| 1 | Bronze — raw made durable | Day 6 ✅ · Day 10 |
| 2 | Silver — raw made trustworthy | **Days 6–7 ✅** |
| 3 | Gold — trustworthy made useful | **Day 7 ✅** |
| 4 | Time travel | Day 10 · Day 24 |
| 5 | Streaming ingestion | Day 12 · Day 13 |
| 6 | Scheduled jobs | Day 14 |
| 7 | Dashboard | Day 15 |
| 8 | Snowflake | Days 16–25 |

Stages 2 and 3 — the cleaning and the aggregation, the analytical heart of the whole
pipeline — are built from Day 6 and Day 7 material. You can already write every line of
both.

Everything else has a date on it. Delta is Day 10. Ingestion is Day 12. Streaming is
Day 13. The scheduled job is Day 14. Snowflake starts Day 16. The two platforms meet on
Day 25. From Day 31 you build the whole thing yourself, on your own data.

---
## Maintenance

Resets the streaming demo to its starting state. Leaves every table intact.

In [ ]:
dbutils.fs.rm(f"{LANDING}/batch_02", recurse=True)
dbutils.fs.rm(CHECKPOINT, recurse=True)
dbutils.fs.rm(SCHEMA_LOC, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {T_STREAM}")
dbutils.fs.mkdirs(CHECKPOINT)
dbutils.fs.mkdirs(SCHEMA_LOC)

run_autoloader()

display(dbutils.fs.ls(LANDING))
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {T_STREAM}"))

In [ ]:
# Full teardown. Uncomment only for a deliberate clean slate.
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
print("inert")